# Multi-Model Climate Forecasting with Genetic Algorithm-Based Model Selection
**CS5100 - Foundations of Artificial Intelligence**
Aniket Nandi, Akshay Ashok Bannatti

---
This notebook runs the full pipeline:
1. Data loading and preprocessing
2. Default (untuned) baseline evaluation
3. GA-based joint model selection + hyperparameter optimisation
4. Random search baseline (equivalent compute budget)
5. Strategy comparison
6. Climate projections to 2050
7. Ablation study on GA components

In [ ]:
# !pip install numpy pandas scikit-learn statsmodels tensorflow matplotlib seaborn pmdarima --quiet

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from data.loaders import load_temperature, load_co2, load_sea_level, train_test_split_ts
from models.statistical import LinearRegressionModel, ARIMAModel, evaluate_all
from models.lstm_model import LSTMModel
from ga.chromosome import Chromosome
from ga.engine import GeneticAlgorithm
from ga.random_search import RandomSearch
from utils.visualise import (
    plot_fitness_curve, plot_type_diversity,
    plot_forecast, plot_comparison, plot_projections
)

os.makedirs('results', exist_ok = True)
print('Imports OK')

## 1. Load and Inspect Data

In [ ]:
df_temp = load_temperature()
df_co2 = load_co2()
df_sealevel = load_sea_level(region = 'global')

print('Temperature :', df_temp.shape, '|', df_temp['date'].min(), '->', df_temp['date'].max())
print('CO2 :', df_co2.shape, '|', df_co2['date'].min(), '->', df_co2['date'].max())
print('Sea Level :', df_sealevel.shape, '|', df_sealevel['date'].min(), '->', df_sealevel['date'].max())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize = (12, 9), sharex = False)

axes[0].plot(df_temp['date'], df_temp['anomaly'], lw = 0.8, color = '#C44E52')
axes[0].set_title('Global Surface Temperature Anomaly (C)')
axes[0].set_ylabel('Anomaly (C)')
axes[0].grid(True, alpha = 0.25)

axes[1].plot(df_co2['date'], df_co2['co2_ppm'], lw = 0.8, color = '#4C72B0')
axes[1].set_title('Atmospheric CO2 Concentration (ppm)')
axes[1].set_ylabel('CO2 (ppm)')
axes[1].grid(True, alpha = 0.25)

axes[2].plot(df_sealevel['date'], df_sealevel['sea_level_mm'], lw = 0.8, color = '#55A868')
axes[2].set_title('Global Mean Sea Level Change (mm)')
axes[2].set_ylabel('Sea Level (mm)')
axes[2].grid(True, alpha = 0.25)

plt.tight_layout()
plt.savefig('results/raw_data.png', dpi = 150)
plt.show()

## 2. Default Baseline Evaluation
Evaluate LR, ARIMA(1,1,1), and LSTM with untuned default hyperparameters.

In [ ]:
def run_defaults(series, test_size):
    results = {}
    lr = LinearRegressionModel(look_back = 12, alpha = 1.0)
    ar = ARIMAModel(p = 1, d = 1, q = 1)
    lstm = LSTMModel(look_back = 12, n_layers = 1, units = 64, epochs = 50)
    results['Default LR'] = lr.walk_forward_rmse(series, test_size)
    results['Default ARIMA'] = ar.walk_forward_rmse(series, test_size)
    results['Default LSTM'] = lstm.walk_forward_rmse(series, test_size)
    return results

TARGET = 'temperature'
series = df_temp['anomaly'].values.astype(float)
dates = df_temp['date'].values
TEST_SIZE = max(12, int(len(series) * 0.15))

defaults = run_defaults(series, TEST_SIZE)
for k, v in defaults.items():
    print(f'{k:<20} RMSE = {v:.4f}')

## 3. Genetic Algorithm Optimisation

In [ ]:
# GA Parameters

POP_SIZE = 30
N_GENS = 50
SEED = 42

ga = GeneticAlgorithm(
    series = series,
    test_size = TEST_SIZE,
    pop_size = POP_SIZE,
    n_generations = N_GENS,
    seed = SEED,
    verbose = True,
)
best_ga, ga_log = ga.run()

print('\nBest GA chromosome:')
print(best_ga)

In [ ]:
fig = plot_fitness_curve(ga_log, title = f'{TARGET.capitalize()} – GA Convergence')
plt.savefig(f'results/{TARGET}_fitness.png', dpi=150)
plt.show()

fig = plot_type_diversity(ga_log, title = f'{TARGET.capitalize()} – Population Diversity')
plt.savefig(f'results/{TARGET}_diversity.png', dpi = 150)
plt.show()

## 4. Random Search Baseline (Equivalent Budget)

In [ ]:
N_EVALS = POP_SIZE * N_GENS

rs = RandomSearch(
    series = series,
    test_size = TEST_SIZE,
    n_evals = N_EVALS,
    seed = SEED + 1,
    verbose = False,
)
best_rs, rs_log = rs.run()

print('Best Random Search chromosome:')
print(best_rs)

## 5. Strategy Comparison

In [ ]:
comparison = {**defaults, 'GA': best_ga.rmse, 'Random Search': best_rs.rmse}

print('\nStrategy Comparison')
for k, v in sorted(comparison.items(), key=lambda x: x[1]):
    flag = ' <- best' if v == min(comparison.values()) else ''
    print(f'{k:<25}  RMSE = {v:.4f}{flag}')

fig = plot_comparison(comparison, title = f'{TARGET.capitalize()} – Strategy Comparison')
plt.savefig(f'results/{TARGET}_comparison.png', dpi = 150)
plt.show()

## 6. Forecast vs Actuals (Best GA Model)

In [ ]:
# Fit best GA model on train, predict test window

best_model = best_ga.build_model()
best_model.fit(series[:-TEST_SIZE])
preds = best_model.predict(TEST_SIZE)

metrics = evaluate_all(series[-TEST_SIZE:], preds)
print(f'Test RMSE : {metrics["rmse"]:.4f}')
print(f'Test MAE : {metrics["mae"]:.4f}')
print(f'Test MAPE : {metrics["mape"]:.2f}%')

fig = plot_forecast(
    dates = dates,
    actuals = series,
    predictions = preds,
    title = f'{TARGET.capitalize()} – GA Best ({best_ga.model_type})',
    ylabel = 'Temperature Anomaly (C)',
)
plt.savefig(f'results/{TARGET}_forecast.png', dpi=150)
plt.show()

## 7. Climate Projections 2025 – 2050

In [ ]:
from main import project_future

N_PROJ = 25 * 12   # 25 years x 12 months
proj, ci_lo, ci_hi = project_future(best_ga, series, n_steps = N_PROJ, n_bootstrap = 50)

last_date = pd.Timestamp(dates[-1])
future_dates = pd.date_range(
    last_date + pd.DateOffset(months = 1), periods = N_PROJ, freq = 'MS'
)

fig = plot_projections(
    historical_dates = dates,
    historical_values = series,
    future_dates = future_dates,
    future_values = proj,
    ci_lower = ci_lo,
    ci_upper = ci_hi,
    title = f'{TARGET.capitalize()} Projection 2025 – 2050 (GA-optimised {best_ga.model_type})',
    ylabel = 'Temperature Anomaly (C)',
)
plt.savefig(f'results/{TARGET}_projection.png', dpi = 150)
plt.show()

print(f'Projected value in 2050: {proj[-1]:.4f}')

## 8. Ablation Study
Vary one GA component at a time and measure final best RMSE.

In [ ]:
ablation_results = {}

# Vary population size
for pop in [10, 20, 30, 50]:
    ga_abl = GeneticAlgorithm(series, test_size = TEST_SIZE,
                              pop_size = pop, n_generations = 20,
                              seed = SEED, verbose = False)
    best_abl, _ = ga_abl.run()
    ablation_results[f'pop = {pop}'] = best_abl.rmse
    print(f'pop = {pop:<3} best RMSE = {best_abl.rmse:.4f}')

# Vary mutation rate
for mut in [0.05, 0.10, 0.20, 0.40]:
    ga_abl = GeneticAlgorithm(series, test_size = TEST_SIZE,
                              pop_size = 20, n_generations = 20,
                              mut_rate = mut, seed = SEED, verbose = False)
    best_abl, _ = ga_abl.run()
    ablation_results[f'mut = {mut}'] = best_abl.rmse
    print(f'mut = {mut} best RMSE = {best_abl.rmse:.4f}')

# Vary crossover rate
for cx in [0.3, 0.5, 0.7, 0.9]:
    ga_abl = GeneticAlgorithm(series, test_size = TEST_SIZE,
                              pop_size = 20, n_generations = 20,
                              cx_prob = cx, seed = SEED, verbose = False)
    best_abl, _ = ga_abl.run()
    ablation_results[f'cx = {cx}'] = best_abl.rmse
    print(f'cx = {cx} best RMSE = {best_abl.rmse:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize = (10, 4))
names = list(ablation_results.keys())
values = [ablation_results[n] for n in names]
ax.bar(names, values, color = '#4C72B0', edgecolor = 'white')
ax.set_ylabel('Best RMSE')
ax.set_title('Ablation Study - GA Component Sensitivity')
ax.tick_params(axis = 'x', rotation = 45)
ax.grid(True, axis = 'y', alpha = 0.3)
plt.tight_layout()
plt.savefig('results/ablation.png', dpi = 150)
plt.show()

## 9. Run All Three Indicators
Set `RUN_ALL = True` to execute the full pipeline for temperature, CO2, and sea level.

In [ ]:
RUN_ALL = False

if RUN_ALL:
    from main import run_indicator
    for ind in ['temperature', 'co2']:
        run_indicator(ind, pop_size = 30, n_generations = 50, seed = 42, verbose = False)
    for reg in ['global', 'indian_ocean', 'bay_of_bengal', 'arabian_sea']:
        run_indicator('sea_level', pop_size = 30, n_generations = 50,
                      seed = 42, region = reg, verbose = False)
    print('All indicators complete. See results/')